# 1) Create source sound collection

This notebook includes the code to create the collection of sounds that will later be used as source material for our audio mosaicing application. The collection of sounds is created by defining a number of queries to be performed using the Freesound API and concatenanting the results of each query. A number of metadata fields are stored for each sound in the collection and saved into a Pandas DataFrame object and CSV file in disk. For each sound in the collection, we also download an OGG preview and store it in disk.

This notebook uses the `freesound` Python package for interacting with the Freesound API. The source code for this package can be found here: https://github.com/mtg/freesound-python. In this repository you'll find a Python script with [examples](https://github.com/MTG/freesound-python/blob/master/examples.py) to learn how to interact with the API. Nevertheless, if you are further interested in the Freesound API, check the [API documentation](http://freesound.org/docs/api/) which provides more information.

**NOTE**: A Freesound API key is provided in this notebook, but you should make a Freesound account and get your own key. You can get a key here: https://freesound.org/apiv2/apply/

In [10]:
%pip install essentia -q
%pip install git+https://github.com/mtg/freesound-python.git -q
from google.colab import drive
drive.mount('/content/drive')

  Cloning https://github.com/mtg/freesound-python.git to /private/var/folders/hg/08z3tgbj2cj4qdxty9pxbfgr0000gn/T/pip-req-build-c0vfg1nn
  Running command git clone --filter=blob:none --quiet https://github.com/mtg/freesound-python.git /private/var/folders/hg/08z3tgbj2cj4qdxty9pxbfgr0000gn/T/pip-req-build-c0vfg1nn
  Resolved https://github.com/mtg/freesound-python.git to commit 73cf6d14f7ce8174d943bdc78ff30f99878a5db8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
%cd /content

/Users/martinsssssss/Desktop/SMC/ACTSM/Block II/1. Sound Retrieval with Fresound/audio-mosaicing


/Users/martinsssssss/opt/anaconda3/envs/audio-mosaicing/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [15]:
import os
import pandas as pd
import numpy as np
import freesound
from IPython.display import display

FREESOUND_API_KEY = 'MWlym7kxqnyVRbmTtWJSRTCXlemiu4ocYZurgCkD'  # Please replace by your own Freesound API key
FILES_DIR = 'files'  # Place where to store the downloaded diles. Will be relative to the current folder.
DATAFRAME_FILENAME = 'dataframe.csv'  # File where we'll store the metadata of our sounds collection
FREESOUND_STORE_METADATA_FIELDS = ['id', 'name', 'username', 'previews', 'license', 'tags']  # Freesound metadata properties to store

freesound_client = freesound.FreesoundClient()
freesound_client.set_token(FREESOUND_API_KEY)
if not os.path.exists(FILES_DIR): os.mkdir(FILES_DIR)

In [16]:
# Define some util functions

def query_freesound(query, filter, num_results=10):
    """Queries freesound with the given query and filter values.
    If no filter is given, a default filter is added to only get sounds shorter than 30 seconds.
    """
    if filter is None:
        filter = 'duration:[0 TO 30]'  # Set default filter
    pager = freesound_client.search(
        query = query,
        filter = filter,
        fields = ','.join(FREESOUND_STORE_METADATA_FIELDS),
        group_by_pack = 1,
        page_size = num_results
    )
    return [sound for sound in pager]

def retrieve_sound_preview(sound, directory):
    """Download the high-quality OGG sound preview of a given Freesound sound object to the given directory.
    """
    return freesound.FSRequest.retrieve(
        sound.previews.preview_hq_ogg,
        freesound_client,
        os.path.join(directory, sound.previews.preview_hq_ogg.split('/')[-1])
    )

def make_pandas_record(sound):
    """Create a dictionary with the metadata that we want to store for each sound.
    """
    record = {key: sound.as_dict()[key] for key in FREESOUND_STORE_METADATA_FIELDS}
    del record['previews']  # Don't store previews dict in record
    record['freesound_id'] = record['id']  # Rename 'id' to 'freesound_id'
    del record['id']
    record['path'] = "files/" + sound.previews.preview_hq_ogg.split("/")[-1]  # Store path of downloaded file
    return record

In [14]:
# ── Collection 1: Dark Techno / Acid ─────────────────────────────────────────

# Our collection of sounds is made by appending the results of a number of different queries to freesound
# The query terms, query filters and the number of results per query are all defined here.
# Information about how to define filters can be found in the Freesound API documentation: https://freesound.org/docs/api/resources_apiv2.html#request-parameters-text-search-parameters
freesound_queries = [
    {
        'query': 'kick drum',
        'filter': 'duration:[0 TO 2]',
        'num_results': 15,
    },
    {
        'query': 'dark techno percussion',
        'filter': 'duration:[0 TO 2]',
        'num_results': 15,
    },
    {
        'query': 'industrial metal clang',
        'filter': 'duration:[0 TO 2]',
        'num_results': 10,
    },
    {
        'query': 'synthesizer bass dark',
        'filter': 'duration:[0 TO 3]',
        'num_results': 10,
    },
    {
        'query': 'noise drone dark',
        'filter': 'duration:[0 TO 3]',
        'num_results': 13,
    },
    {
        'query': 'acid bass 303',
        'filter': 'duration:[0 TO 3]',
        'num_results': 8,
    },
    {
        'query': 'hi-hat electronic',
        'filter': 'duration:[0 TO 1]',
        'num_results': 8,
    },
    {
        'query': 'clap snare electronic',
        'filter': 'duration:[0 TO 1]',
        'num_results': 8,
    },
]

# Do all queries and concatenate the results in a single list of sounds
sounds = sum([query_freesound(query['query'], query['filter'], query['num_results']) for query in freesound_queries],[])

# Download the sounds and save them to FILES_DIR folder
for count, sound in enumerate(sounds):
    print('Downloading sound with id {0} [{1}/{2}]'.format(sound.id, count + 1, len(sounds)))
    retrieve_sound_preview(sound, 'files/')

# Make a Pandas DataFrame with the metadata of our sound collection and save it
df =  pd.DataFrame([make_pandas_record(s) for s in sounds])
df.to_csv(DATAFRAME_FILENAME)
print('Saved DataFrame with {0} entries! {1}'.format(len(df), DATAFRAME_FILENAME))

# Show the contents of our DataFrame (the metadata of our source collection)
display(df)

Saved DataFrame with 63 entries! dataframe.csv


,name,username,license,tags,freesound_id,path
0,Kick Drum E-1,Rob10,http://creativecommons.org/publicdomain/zero/1.0/,"[Basic, Drum, Kick, one, sample, shot]",132584,files/132584_2409787-hq.ogg
1,Clicky Sub Kick Drum,TheEndOfACycle,http://creativecommons.org/publicdomain/zero/1.0/,"[1-hit, drum, drums, hit, kick, layered, one-s...",673512,files/673512_3130497-hq.ogg
2,Lo fi kick drum 01.wav,johnnypanic,http://creativecommons.org/publicdomain/zero/1.0/,"[drum, drums, kick, lofi, percussion, sample]",647617,files/647617_24119-hq.ogg
3,Subby Kick Drum,Mattc90,http://creativecommons.org/publicdomain/zero/1.0/,"[amazing, awesome, beat, bpm, campbell, cool, ...",400707,files/400707_1126957-hq.ogg
4,Fractanimal_Acoustic_Drum_Kit_Kick_7.wav,johnnydekk,http://creativecommons.org/publicdomain/zero/1.0/,"[drum, kick, kit, percussion]",581461,files/581461_5790048-hq.ogg
...,...,...,...,...,...,...
58,clap02.wav,esformouse,http://creativecommons.org/licenses/sampling+/...,"[clap, distortion, drum, electronic, percussio...",56808,files/56808_231832-hq.ogg
59,BleepdrumBD_02,oceansonmars,http://creativecommons.org/publicdomain/zero/1.0/,"[Analog, Bleep, Circuit-Bend, Clap, Drum, Drum...",827848,files/827848_15225418-hq.ogg
60,Dirty electro bass.wav,staticpony1,http://creativecommons.org/publicdomain/zero/1.0/,"[4x4-Records, Bass, Beat, Clap, Closed, Dance,...",249592,files/249592_4508519-hq.ogg
61,Trap Snare 2,muffin3k,http://creativecommons.org/publicdomain/zero/1.0/,"[drum, drums, hiphop, rap, sample, snare, trap]",517297,files/517297_10137963-hq.ogg


## Collection 2: Piano Source

A contrasting source collection based on **piano sounds**.
Using piano as source for audio mosaicing produces very different results vs dark techno:
- Piano has wide timbral variety (notes, dynamics, attack/decay shapes)
- The contrast between techno target and piano source creates an experimental texture
- Comparing both collections is one of the key analyses for the report.

In [17]:
# ── Collection 2: Piano ───────────────────────────────────────────────────────
DATAFRAME_PIANO_FILENAME = 'dataframe_piano.csv'

piano_queries = [
    {'query': 'piano note single',  'filter': 'duration:[0 TO 3]', 'num_results': 20},
    {'query': 'piano chord',        'filter': 'duration:[0 TO 4]', 'num_results': 15},
    {'query': 'piano keys',         'filter': 'duration:[0 TO 2]', 'num_results': 15},
    {'query': 'piano low bass note','filter': 'duration:[0 TO 3]', 'num_results': 10},
]

piano_sounds = sum([
    query_freesound(q['query'], q['filter'], q['num_results'])
    for q in piano_queries
], [])

for count, sound in enumerate(piano_sounds):
    print(f'Downloading piano sound {sound.id} [{count+1}/{len(piano_sounds)}]')
    retrieve_sound_preview(sound, 'files/')

df_piano = pd.DataFrame([make_pandas_record(s) for s in piano_sounds])
df_piano.to_csv(DATAFRAME_PIANO_FILENAME)
print(f'Saved piano DataFrame with {len(df_piano)} entries → {DATAFRAME_PIANO_FILENAME}')
display(df_piano)

Saved piano DataFrame with 43 entries → dataframe_piano.csv


,name,username,license,tags,freesound_id,path
0,g6 note,rondindon,http://creativecommons.org/publicdomain/zero/1.0/,"[note, octave, piano, single, singlenote]",740920,files/740920_16083839-hq.ogg
1,Casio 1000P Preset - Piano C.wav,acollier123,https://creativecommons.org/licenses/by/4.0/,"[Additive, CT1000, Casio, casiotone, piano, sa...",315694,files/315694_2076119-hq.ogg
2,"Piano, C# -2, Muffled, Single Note",VizAion,http://creativecommons.org/publicdomain/zero/1.0/,"[Ab-note, Audio-Sample, Deep-sound, Keys, Musi...",795601,files/795601_17061405-hq.ogg
3,Upright Piano Remeau 23 C8,Sadiquecat,http://creativecommons.org/publicdomain/zero/1.0/,"[8, Average-piano, C, C8, FR-AV2, Living-room,...",794382,files/794382_5287430-hq.ogg
4,D7.ogg,TEDAgame,http://creativecommons.org/publicdomain/zero/1.0/,"[complete, keys, notes, old, piano, reverb, su...",448617,files/448617_9311684-hq.ogg
5,Slide Whistle in a Tin Can.wav,gurdonark,http://creativecommons.org/licenses/by/3.0/,"[mill, note, patch, piercing, sample, slide, s...",161641,files/161641_59021-hq.ogg
6,Kalimba_note4.wav,radian,http://creativecommons.org/licenses/by/3.0/,"[kalimba, metallic, thumb-piano]",25049,files/25049_76945-hq.ogg
7,Yamaha CS-30L - Toy Piano - C5 (Small Plick-72...,modularsamples,http://creativecommons.org/publicdomain/zero/1.0/,"[C5, Yamaha-CS-30L, midi-note-72, multisample,...",314965,files/314965_2050105-hq.ogg
8,Dog single howl,Jace,http://creativecommons.org/publicdomain/zero/1.0/,"[barking, beagle, dog, howl, nose, pooch, pupp...",155322,files/155322_60285-hq.ogg
9,"Vivace, con screamo - No Solo.wav",isaiah.chentnik,http://creativecommons.org/licenses/by/3.0/,"[bartok, chentnik, con, county, fair, isaiah, ...",52765,files/52765_634268-hq.ogg
